# PACE-ASD: single-clip inference walkthrough

This notebook runs the released inference API on one pre-extracted landmark sequence and
shows every intermediate output the software exposes: the normalised landmarks, the
derived velocity and acceleration, the Block-ESG block scores and selected blocks, the
calibrated probability, and gradient x input attribution summaries.

Outputs are research outputs from a model trained on one small single-site dataset of
children walking towards a camera. They are not diagnostic statements.

In [ ]:
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt
import torch

repo_root = os.path.abspath('..' if os.path.exists('../src') else '.')
sys.path.insert(0, os.path.join(repo_root, 'src'))
from inference_api import PACEASDPredictor

CLIP = os.path.join(repo_root, 'processed', 'features', 'asd_45.npy')
# v1.1 checkpoint (padding-masked). The released v1.0 checkpoints in models/A1 also load.
CKPT = os.path.join(repo_root, 'models_v11', 'A1', 'fold1_seed42.pt')
if not os.path.exists(CKPT):
    CKPT = os.path.join(repo_root, 'models', 'A1', 'fold1_seed42.pt')
CFG = os.path.join(repo_root, 'configs', 'inference.yaml')
print('checkpoint:', os.path.relpath(CKPT, repo_root))

## 1. Input: normalised landmark sequence
Shape `(300, 33, 2)`: mid-hip centred, divided by shoulder width, zero-padded after the last frame.

In [ ]:
x = np.load(CLIP).astype(np.float32)
valid = np.abs(x).sum(axis=(1, 2)) > 1e-4
print('shape', x.shape, '| frames with a detected pose:', int(valid.sum()))

## 2. Kinematic descriptor
Velocity and acceleration are computed inside the model exactly as below (finite differences, zeroed across invalid frames, scaled by 10 and 5).

In [ ]:
vel = np.zeros_like(x); acc = np.zeros_like(x)
vv = valid[1:] & valid[:-1]
vel[1:][vv] = (x[1:] - x[:-1])[vv] * 10.0
va = valid[2:] & valid[1:-1] & valid[:-2]
acc[2:][va] = (vel[2:] - vel[1:-1])[va] * 5.0

n = int(valid.sum()) + 10
fig, axes = plt.subplots(3, 1, figsize=(9, 6), sharex=True)
for ax, arr, name in zip(axes, (x, vel, acc), ('position', 'velocity (x10)', 'acceleration (x50)')):
    ax.plot(arr[:n, 15, 0], label='left wrist, x'); ax.plot(arr[:n, 27, 0], label='left ankle, x')
    ax.set_ylabel(name)
axes[0].legend(loc='upper right'); axes[-1].set_xlabel('frame (30 frames/s)')
plt.tight_layout(); plt.show()

## 3. Model inference
`PACEASDPredictor` rebuilds the architecture from the configuration stored in the checkpoint and restores its Platt scaler.

In [ ]:
predictor = PACEASDPredictor(checkpoint=CKPT, config=CFG, device='cpu')
result = predictor.predict_npy(CLIP)
print('mask_padding (v1.1):', predictor.model.mask_padding)
for k in ('raw_probability', 'calibrated_probability', 'prediction'):
    print(f'{k:24s}', result[k])

## 4. Block-ESG: block scores and selected blocks
Blocks are 15 frames (0.5 s). `valid_frames = 0` marks a padding block; v1.1 checkpoints only select padding when the clip has fewer than 8 valid blocks, and the Transformer ignores padded tokens.

In [ ]:
scores = np.array(result['event_saliency'])
selected = {e['start_frame'] // 15 for e in result['selected_events']}
n_valid_blocks = int(np.ceil(valid.sum() / 15))
colors = ['tab:orange' if b in selected else ('tab:blue' if b < n_valid_blocks else 'lightgrey')
          for b in range(len(scores))]
plt.figure(figsize=(9, 3))
plt.bar(range(len(scores)), scores, color=colors, edgecolor='black')
plt.xlabel('block index (15 frames each)'); plt.ylabel('gate score')
plt.title('orange: selected | blue: valid, not selected | grey: padding')
plt.tight_layout(); plt.show()
for e in result['selected_events']:
    print(e)

## 5. Attribution summaries
Gradient x input on the 198-d per-frame descriptor, summed over frames with a detected pose. These describe what the trained model is sensitive to for this clip; they are not measurements of the child's movement.

In [ ]:
print(json.dumps({k: result[k] for k in ('kinematic_stream_attribution',
                                         'body_region_attribution',
                                         'body_region_attribution_motion')}, indent=2))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3))
s = result['kinematic_stream_attribution']; r = result['body_region_attribution_motion']
a1.bar(list(s), list(s.values()), color='tab:gray'); a1.set_title('stream share')
a2.bar(list(r), list(r.values()), color='tab:gray'); a2.set_title('region share (velocity + acceleration)')
plt.tight_layout(); plt.show()

## 6. Raw video
`predictor.predict_video(path)` runs MediaPipe Pose (model complexity 2) on each frame and applies the same normalisation before inference. Videos are not distributed with the repository.